In [ ]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

url = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/ufo-scrubbed-geocoded-time-standardized-00.csv"
df = pd.read_csv(url)
df

### Plot 1: Distribution of Reported UFO Shapes

In [ ]:
print(df.columns)

In [ ]:
column_names = [
    'datetime', 'city', 'state', 'country', 'shape',
    'duration_seconds', 'duration_described', 'comments',
    'date_posted', 'latitude', 'longitude'
]

if df.shape[1] == len(column_names):
    df.columns = column_names
else:
    df = pd.read_csv(url, names=column_names, skiprows=1, low_memory=False)

df['duration_seconds'] = pd.to_numeric(df['duration_seconds'], errors='coerce')
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
df['datetime_parsed'] = pd.to_datetime(df['datetime'].str.replace('24:00', '00:00'), format='%m/%d/%Y %H:%M', errors='coerce')
df['year'] = df['datetime_parsed'].dt.year

display(df.head())

In [ ]:
# Plot 1: Distribution of UFO Shapes
shape_chart = alt.Chart(df.dropna(subset=['shape'])).mark_bar().encode(
    x=alt.X('count():Q', title='Number of Sightings'),
    y=alt.Y('shape:N', sort='-x', title='UFO Shape'),
    color=alt.Color('count():Q', scale=alt.Scale(scheme='viridis'), legend=alt.Legend(title="Sighting Count"))).properties(
    width=600,
    height=400,
    title="Frequency of Reported UFO Shapes")
shape_chart

In [ ]:
import numpy as np

df_clean = df.dropna(subset=['year', 'duration_seconds']).copy()
df_clean = df_clean[(df_clean['year'] >= 1940) & (df_clean['duration_seconds'] > 0)]
df_clean['log_duration'] = np.log10(df_clean['duration_seconds'])

brush = alt.selection_interval(encodings=['x'])

detail_chart = alt.Chart(df_clean).mark_circle(opacity=0.4, size=25).encode(
    x=alt.X('year:O', title='Year of Sighting', axis=alt.Axis(values=list(range(1940, 2015, 2)))),
    y=alt.Y('log_duration:Q', title='Log10 Duration (seconds)'),
    color=alt.condition(brush, alt.Color('country:N', scale=alt.Scale(scheme='category10'), title='Country'), alt.value('lightgray')),
    tooltip=['city', 'state', 'shape', 'duration_described']).properties(
    width=600,
    height=300,
    title="UFO Sighting Durations Over Time)").transform_filter(brush)

timeline_selector = alt.Chart(df_clean).mark_bar().encode(
    x=alt.X('year:O', title='Drag to Select Year Range', axis=alt.Axis(values=list(range(1940, 2015, 2)))),
    y=alt.Y('count():Q', title='Count'),
    color=alt.value('darkblue')).add_params(brush).properties(
    width=600,
    height=80,
    title="Drag/Select Years to Filter Scatter Plot")
interactive_dashboard = alt.vconcat(detail_chart, timeline_selector)
interactive_dashboard

In [ ]:
shape_chart.save('ufo_shapes.json')
interactive_dashboard.save('ufo_interactive_timeline.json')
print("saved to JSON")